# Historische bal-v2-training en de 80 gereviewde beelden
Versie 5, 15 november 2026. Dit notebook controleert de verbinding met de bestaande training. De 80 ongewijzigde reviewbeelden vormen een subset van de 132 bronbeelden achter 282 crops. Dit is geen afzonderlijke nieuwe run op exact 80 beelden. Alleen de huidige 80 zijn later door het team goedgekeurd. De cijfers hieronder gelden voor de oorspronkelijke volledige validatie, niet voor een afzonderlijke nieuwe meting op de 32 review-validatiebeelden of Andrews huidige balmodel.

De oorspronkelijke data, begin- en eindgewichten, args.yaml en results.csv zijn meegeleverd. Standaard worden alleen bestanden gelezen. RUN_TRAINING staat uit.

In [1]:
from pathlib import Path
import runpy, json, csv
ROOT = Path.cwd().resolve()
if not (ROOT/'scripts/controle_labelset_training.py').exists():
    ROOT = ROOT/'03_Code'
assert (ROOT/'scripts/controle_labelset_training.py').exists()
audit = runpy.run_path(str(ROOT/'scripts/controle_labelset_training.py'))['audit']
result = audit(ROOT)
print(json.dumps({k:v for k,v in result.items() if k!='frames'}, indent=2))

{
  "checked_review_frames": 80,
  "split": {
    "train": 48,
    "val": 32
  },
  "positive_crops_from_review_set": 174,
  "full_training_dataset_crops": 282,
  "full_source_images": 132,
  "dataset_fingerprint": "afa5bbce1eff649d9967e927d0d54f36671dcea3c855620b7919d0a926870d2e",
  "candidate_weights_sha256": "0bfd52b880a6994b646c78374fb00156ba211b3a464f8c8a273873f9b5d2bb52",
  "historical_v2_full_mAP50": 0.4163668559634949,
  "historical_v2_full_mAP50_95": 0.18008868890445887,
  "historical_v2_tiled_mAP50": 0.5748519958889426,
  "historical_v2_tiled_mAP50_95": 0.23336801171460783,
  "meaning": "The approved 80-frame set is an unchanged subset of the corpus used for historical v2 training/validation. No separate new run on exactly 80 frames.",
  "limits": "Review approval is later than the historical run. Other source images are not thereby human-approved. Metrics cover the full historical validation, not an isolated 32-frame re-evaluation or the deployed Andrew model.",
  "new_train

## Vastgelegde training en evaluatie
De oorspronkelijke fine-tuning gebruikt F005 als startgewicht, twaalf epochs, 640 pixels, batch vier, freeze tien en AdamW met learning rate 0,0001. De COCO-klasse 32 vertegenwoordigt de bal. De opgeslagen evaluatie zet die uitvoer om naar de baltaak. AP@50 en AP@50:95 hieronder komen uit het bestaande ontwikkelrapport. De tegelroute gebruikt een andere invoerroute dan evaluatie op volledige beelden; verwissel die scores niet.

In [2]:
experiment=json.loads((ROOT/'reports/broadcast-data/v2-experiment.json').read_text(encoding='utf-8'))
print('Historische parameters:', experiment['preregistration']['training'])
with (ROOT/'artifacts/broadcast-ball-v2/fine-tune/results.csv').open(encoding='utf-8-sig') as f:
    logs=list(csv.DictReader(f))
print('Opgeslagen epochs:',len(logs))
assert len(logs)==12
for route in ['full','tiled']:
    measurement=experiment['candidate'][route]
    print(route, 'mAP50:',measurement['mAP50'],'mAP50:95:',measurement['mAP50_95'])

Historische parameters: {'epochs': 12, 'imgsz': 640, 'batch': 4, 'freeze': 10, 'optimizer': 'AdamW', 'lr0': 0.0001, 'lrf': 0.1, 'cos_lr': True, 'warmup_epochs': 3, 'weight_decay': 0.0005, 'patience': 12, 'amp': True, 'workers': 0, 'seed': 42, 'deterministic': True, 'cache': False, 'mosaic': 0.0, 'mixup': 0.0, 'copy_paste': 0.0, 'degrees': 0.0, 'scale': 0.0, 'translate': 0.0, 'fliplr': 0.5, 'flipud': 0.0, 'hsv_h': 0.01, 'hsv_s': 0.2, 'hsv_v': 0.15, 'plots': False}
Opgeslagen epochs: 12
full mAP50: 0.4163668559634949 mAP50:95: 0.18008868890445887
tiled mAP50: 0.5748519958889426 mAP50:95: 0.23336801171460783


## Optioneel: een nieuwe training met dezelfde opgeslagen instellingen
Deze cel begint pas met trainen wanneer RUN_TRAINING bewust True wordt. Zij bindt de YAML aan deze uitgepakte map en schrijft naar een nieuwe runmap. De historische checkpointbestanden worden niet overschreven. Een geschikte GPU en de geïnstalleerde Ultralytics-omgeving zijn nodig voor device 0; andere software- en hardwareversies kunnen andere uitkomsten opleveren.

In [3]:
RUN_TRAINING = False
if RUN_TRAINING:
    from datetime import datetime
    import yaml
    from ultralytics import YOLO
    dataset=ROOT/'data/broadcast-training-v2/ball-coco'
    spec=yaml.safe_load((dataset/'data.yaml').read_text(encoding='utf-8'))
    spec['path']=str(dataset)
    (ROOT/'.cache').mkdir(exist_ok=True)
    local_yaml=ROOT/'.cache/bal-v2-local.yaml'
    local_yaml.write_text(yaml.safe_dump(spec,sort_keys=False),encoding='utf-8')
    model=YOLO(str(ROOT/'artifacts/fullscene-ball-v1/fullscene_ball_f005.pt'))
    model.train(data=str(local_yaml),device=0,project=str(ROOT/'artifacts/nieuwe-bal-v2-runs'),
                name=datetime.now().strftime('%Y%m%d_%H%M%S'),exist_ok=False,
                **experiment['preregistration']['training'])
else:
    print('Geen nieuwe training gestart.')

Geen nieuwe training gestart.
